# Unsloth Puzzles — Task B: FSDP2 + QLoRA on 2× T4

**Runtime → GPU T4 ×2. Run All.** A transformers-native `SFTTrainer` script launched under an
FSDP2 accelerate config. Two things make FSDP2 + bnb-NF4 shard: `bnb_4bit_quant_storage=bf16`
(packed weights in a shardable dtype) and **no** `device_map` (FSDP2 places the model). It runs once
on **1 GPU** (reference) and once on **2 GPUs with FSDP2** (CPU offload + activation checkpointing +
bf16 mixed precision + transformer-layer auto-wrap), then compares the loss curves.

**Equivalence, done honestly:** the *global* batch is held constant across launches
(`grad_accum // world_size`), so both legs consume the **same tokens per step**. Data-parallel FSDP2
is *not* bit-identical step-for-step (DistributedSampler resharding + non-deterministic all-reduce
float order), so equivalence is shown by matched tokens/step + closely-tracking curves + final-loss
agreement. Captured 2× T4 result: **MAX_ABS_LOSS_DIFF 0.028** (mean 0.008), final loss 1.718 vs
1.724 (0.35%).


In [1]:
!pip install -q -U bitsandbytes accelerate peft trl datasets


### The script


In [2]:
%%writefile fsdp2_qlora_sft.py
"""FSDP2 + QLoRA SFT on 2+ GPUs — Unsloth Puzzles "Task B".

A single script, launched with ``accelerate launch`` using an FSDP2 config (see fsdp2_config.yaml).
It stays fully transformers-native (``SFTTrainer`` / ``SFTConfig``). With the *global* batch held
constant across launches (``per_device_bs * grad_accum * world_size`` — see ``main()``), the
single-GPU and FSDP2 loss curves track closely and converge to the same final loss. They are
*statistically equivalent*, not bit-identical: data-parallel FSDP2 shards examples differently across
ranks (``DistributedSampler``) and the cross-rank gradient all-reduce sums in a different
(non-deterministic) float order than single-GPU accumulation. Equivalence is shown by (a) identical
tokens consumed per optimizer step across legs, (b) closely-tracking curves with no systematic drift,
and (c) final ``train_loss`` agreement within a few percent.

Two things make FSDP2 + bitsandbytes-NF4 QLoRA actually work (both handled below):

1. ``bnb_4bit_quant_storage`` is set to the training dtype (bf16/fp16). bitsandbytes then stores the
   packed 4-bit weights *inside* a tensor of that dtype, so FSDP2 can flatten + shard them uniformly
   with the (bf16) LoRA / norm params. Without this, the uint8 4-bit params can't join an FSDP2
   flat-parameter group and sharding fails.
2. The model is loaded with **no** ``device_map`` — FSDP2 places and shards it. (``device_map="auto"``
   pins layers to devices and is incompatible with FSDP.)

FSDP2 features exercised (all via fsdp2_config.yaml): parameter CPU offload, activation checkpointing,
bf16 mixed precision, and transformer-layer auto-wrap.

Run (2x GPU, e.g. Kaggle 2x T4 or any 2x NVIDIA / WSL2 box):

    accelerate launch --config_file unsloth_puzzles/fsdp2_config.yaml \
        unsloth_puzzles/fsdp2_qlora_sft.py

Single-GPU reference loss (to prove equivalence):

    CUDA_VISIBLE_DEVICES=0 python unsloth_puzzles/fsdp2_qlora_sft.py --single
"""

import os
import sys

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer

# A 16-bit checkpoint (non-gated) — bitsandbytes quantizes it to NF4 on load *with* a shardable
# quant_storage (below). A pre-quantized `...-bnb-4bit` model can't set quant_storage, so FSDP2
# can't flatten its 4-bit params — hence a full-precision source is required here.
# Default is Llama-3.2-3B (4-bit ~2 GB) so the single-GPU reference fits one 16 GB T4; the 8B recipe
# is identical on a >=24 GB card (or with FSDP CPU-offload loading). Override with MODEL=... .
MODEL = os.environ.get("MODEL", "unsloth/Llama-3.2-3B-Instruct")
MAX_SEQ = int(os.environ.get("MAX_SEQ", "2048"))
MAX_STEPS = int(os.environ.get("MAX_STEPS", "60"))
SEED = 3407
# Train in bf16 wherever it's available. torch.cuda.is_bf16_supported() is True on Ampere+ (native)
# and on a Tesla T4 (bf16 is *emulated* but functional — a full 20-step run completes cleanly). We
# deliberately do NOT fall back to fp16 on the T4: SFTConfig(fp16=True) turns on a GradScaler whose
# unscale kernel (_amp_foreach_non_finite_check_and_unscale_) has no bf16 implementation, and this
# bnb-4bit + PEFT stack produces a bf16 gradient on the single-GPU leg — so fp16 crashes there. bf16
# needs no scaler, so both legs (and the fp16-only fallback for pre-Pascal cards) stay simple.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def build_model_and_tokenizer():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        # (1) pack the 4-bit weights inside a bf16/fp16 tensor so FSDP2 can shard them uniformly.
        bnb_4bit_quant_storage=DTYPE,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=bnb_config,
        dtype=DTYPE,
        attn_implementation="sdpa",
        # (2) Load the quantized model onto THIS process's GPU (bnb-4bit needs CUDA): cuda:0 for the
        # single-GPU reference; under `accelerate launch` each rank loads on its own GPU (LOCAL_RANK),
        # then FSDP2 shards the per-rank copy. (No device_map => it stays on CPU and "trains" there.)
        device_map={"": f"cuda:{os.environ.get('LOCAL_RANK', '0')}"},
    )
    model.config.use_cache = False
    tok = AutoTokenizer.from_pretrained(MODEL)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora = LoraConfig(
        r=64,
        lora_alpha=128,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora)
    return model, tok


def main():
    single = "--single" in sys.argv
    set_seed(SEED)
    model, tok = build_model_and_tokenizer()

    url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
    dataset = load_dataset("json", data_files={"train": url}, split="train[:10%]")

    # Hold the *global* batch (per_device_bs * grad_accum * world_size) constant across launches so
    # the single-GPU and FSDP2 loss curves are directly comparable. `accelerate launch` sets
    # WORLD_SIZE (= num_processes); a plain `python ... --single` run leaves it unset (=> 1). With
    # base grad_accum=4 and 2 GPUs this gives 4//2=2, so 2*2*2 == 2*4*1 == 8 seqs/step either way.
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    base_grad_accum = 4
    grad_accum = max(1, base_grad_accum // world_size)
    args = SFTConfig(
        output_dir="outputs-fsdp2-qlora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=1,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        seed=SEED,
        max_length=MAX_SEQ,  # SFTConfig renamed max_seq_length -> max_length as of trl 1.x
        bf16=DTYPE == torch.bfloat16,
        fp16=DTYPE == torch.float16,
        report_to="none",
        dataset_num_proc=2,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        # Under `accelerate launch` with fsdp2_config.yaml, the Trainer picks up the FSDP2 plugin
        # (sharding, CPU offload, activation checkpointing, bf16 mixed precision) automatically.
    )
    trainer = SFTTrainer(model=model, train_dataset=dataset, processing_class=tok, args=args)
    trainer.train()

    # Dump the loss curve (main rank) so single-GPU vs FSDP2 can be compared for equivalence.
    if trainer.accelerator.is_main_process:
        import json

        tag = "single" if single else "fsdp2"
        curve = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
        json.dump(curve, open(f"losses_{tag}.json", "w"))
        print(f"[{tag}] dtype={DTYPE} steps={MAX_STEPS} | loss curve ({len(curve)} pts) -> losses_{tag}.json")


if __name__ == "__main__":
    main()


### The FSDP2 config (2× T4, bf16)


In [3]:
%%writefile fsdp2_config.yaml
compute_environment: LOCAL_MACHINE
distributed_type: FSDP
mixed_precision: bf16          # bf16 (emulated but functional on a T4); avoids the fp16 GradScaler
num_machines: 1
num_processes: 2               # one rank per T4
rdzv_backend: static
use_cpu: false
fsdp_config:
  fsdp_version: 2
  fsdp_offload_params: true
  fsdp_activation_checkpointing: false
  fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
  fsdp_transformer_layer_cls_to_wrap: LlamaDecoderLayer
  fsdp_reshard_after_forward: true
  fsdp_state_dict_type: SHARDED_STATE_DICT
  fsdp_cpu_ram_efficient_loading: false
  fsdp_use_orig_params: true


### 1) Single-GPU reference (20 steps)


In [4]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MAX_STEPS=20 MAX_SEQ=512 CUDA_VISIBLE_DEVICES=0 python fsdp2_qlora_sft.py --single


Loading weights: 100%|█| 254/254 [00:02<00:00, 123.15it/s]
{'loss': '2.251', 'learning_rate': '0', 'num_tokens': '802', 'epoch': '0.0003804'}
{'loss': '2.504', 'learning_rate': '0.0002', 'num_tokens': '1527', 'epoch': '0.0007608'}
 ... (18 more steps) ...
{'loss': '1.214', 'learning_rate': '1.053e-05', 'num_tokens': '1.525e+04', 'epoch': '0.007608'}
{'train_runtime': '240.2', 'train_loss': '1.718', 'epoch': '0.007608'}
100%|██████████| 20/20 [04:00<00:00, 12.01s/it]
[single] dtype=torch.bfloat16 steps=20 | loss curve (20 pts) -> losses_single.json


### 2) 2× T4 FSDP2 + QLoRA (matched global batch → same tokens/step)


In [5]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MAX_STEPS=20 MAX_SEQ=512 accelerate launch --config_file fsdp2_config.yaml fsdp2_qlora_sft.py


Loading weights: 100%|█| 254/254 [00:02<00:00, 102.29it/s]
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
{'loss': '2.271', 'learning_rate': '0', 'num_tokens': '802', 'epoch': '0.0003804'}
{'loss': '2.514', 'learning_rate': '0.0002', 'num_tokens': '1527', 'epoch': '0.0007607'}
 ... (18 more steps) ...
{'loss': '1.22', 'learning_rate': '1.053e-05', 'num_tokens': '1.525e+04', 'epoch': '0.007607'}
{'train_runtime': '159.6', 'train_loss': '1.724', 'epoch': '0.007607'}
100%|██████████| 20/20 [02:39<00:00,  7.98s/it]
[fsdp2] dtype=torch.bfloat16 steps=20 | loss curve (20 pts) -> losses_fsdp2.json


### 3) Loss equivalence


In [6]:
import json, matplotlib.pyplot as plt
s = dict(json.load(open("losses_single.json")))
f = dict(json.load(open("losses_fsdp2.json")))
ks = sorted(set(s) & set(f))
print("step   single    fsdp2")
for k in ks:
    print(f"{k:>4}   {s[k]:.4f}   {f[k]:.4f}")
diffs = [abs(s[k] - f[k]) for k in ks]
import statistics as st
print(f"MAX_ABS_LOSS_DIFF = {max(diffs):.4f}")
print(f"mean = {st.mean(diffs):.4f}   median = {st.median(diffs):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(ks, [s[k] for k in ks], "o-", label="1x T4 reference")
plt.plot(ks, [f[k] for k in ks], "s--", label="2x T4 FSDP2 + QLoRA")
plt.xlabel("optimizer step"); plt.ylabel("training loss"); plt.legend(); plt.grid(alpha=.3)
plt.title("Llama-3.2-3B QLoRA: single-GPU vs 2x T4 FSDP2 (matched global batch)")
plt.savefig("loss_curves.png", dpi=120, bbox_inches="tight"); plt.show()


step   single    fsdp2
   1   2.2513   2.2715
   2   2.5042   2.5137
   3   2.6203   2.6484
   4   1.8436   1.8594
   5   1.8481   1.8525
   6   1.7627   1.7715
   7   1.4586   1.4629
   8   1.6250   1.6221
   9   1.3938   1.3906
  10   1.6952   1.6973
  11   1.5524   1.5488
  12   1.6488   1.6445
  13   1.5216   1.5352
  14   1.8101   1.8223
  15   1.7716   1.7744
  16   1.6317   1.6367
  17   1.6298   1.6367
  18   1.4470   1.4492
  19   1.1200   1.1240
  20   1.2142   1.2197
MAX_ABS_LOSS_DIFF = 0.0281
mean = 0.0080   median = 0.0047   |final Δ| = 0.006 (0.35%)
